# RetinaSafe — Cross-dataset transfer: Kermany → OCTDL  (v2 — hardcoded paths)

**Why v2:** the previous version used recursive `glob` over `/kaggle/input/**` which took 572s walking 86k+ files. This version hardcodes the verified paths and walks only OCTDL.

**Required inputs (attach via right sidebar → + Add Input):**
1. `paultimothymooney/kermany2018` — for in-distribution sanity
2. `orvile/octdl-optical-coherence-tomography-dataset` — OCTDL
3. **Notebook Output:** `samarthmishra208/kermany-baseline` — gives `best.pt` + `kermany_index.csv`

**Settings:** GPU T4 x2, Internet off (no weight download needed). Total runtime ~10 min.

## Cell 1 — Imports + hardcoded paths + verify

In [ ]:
import os, sys, json, time, random, pathlib
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve, average_precision_score

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__, "GPU:", torch.cuda.is_available(),
      "Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Hardcoded paths (verified from /kaggle/input/datasets/ listing)
OCTDL_ROOT = "/kaggle/input/datasets/orvile/octdl-optical-coherence-tomography-dataset"
CKPT = "/kaggle/input/notebooks/samarthmishra208/kermany-baseline/results/best.pt"
KERMANY_INDEX = "/kaggle/input/notebooks/samarthmishra208/kermany-baseline/kermany_index.csv"

for label, p in [("OCTDL_ROOT", OCTDL_ROOT), ("CKPT", CKPT), ("KERMANY_INDEX", KERMANY_INDEX)]:
    assert os.path.exists(p), f"{label} missing at {p}"
    print(f"OK  {label}: {p}")

## Cell 2 — Locate the OCTDL metadata + images

Walks OCTDL only (not all of /kaggle/input/), so it's fast.

In [ ]:
# ---- Print the full structure of OCTDL_ROOT so we can see what's actually there ----
print(f"=== Tree under {OCTDL_ROOT} ===")
all_files = []
all_dirs = []
for root, dirs, files in os.walk(OCTDL_ROOT):
    rel = os.path.relpath(root, OCTDL_ROOT)
    depth = 0 if rel == "." else rel.count(os.sep) + 1
    if depth <= 2:
        print(f"{'  '*depth}{os.path.basename(root) if rel != '.' else os.path.basename(OCTDL_ROOT)}/")
        for f in sorted(files)[:5]:
            print(f"{'  '*(depth+1)}{f}")
        if len(files) > 5:
            print(f"{'  '*(depth+1)}... ({len(files)} files total)")
    for f in files:
        all_files.append(os.path.join(root, f))
    for d in dirs:
        all_dirs.append(os.path.join(root, d))

print(f"\nTotal files under OCTDL_ROOT: {len(all_files):,}")
print(f"Total subdirs: {len(all_dirs):,}")

# Find metadata CSV — case-insensitive, anywhere
META_CSV = None
for fp in all_files:
    if os.path.basename(fp).lower() == "octdl_metadata.csv":
        META_CSV = fp
        break

# Find candidate image directory roots — folders whose direct children are images
img_files_all = [fp for fp in all_files if fp.lower().endswith((".jpg", ".jpeg", ".png"))]
print(f"Image files (jpg/jpeg/png) under OCTDL_ROOT: {len(img_files_all):,}")

if META_CSV:
    print(f"\nFOUND metadata: {META_CSV}")
else:
    print(f"\nMETADATA CSV NOT FOUND — will fall back to directory-based class inference.")

# Sample 5 image paths so we can see the naming convention
print("\nSample image paths:")
for p in img_files_all[:5]:
    print(" ", p)


## Cell 3 — Build OCTDL eval index

Match metadata filenames to actual paths, map disease → binary referable label (Normal = 0, all 6 disease classes = 1). For pure transfer eval, all OCTDL = the test set, no splits.

In [ ]:
import re

# ---- Build OCTDL metadata, either from CSV or from directory structure ----
if META_CSV:
    meta = pd.read_csv(META_CSV)
    print(f"Loaded {META_CSV}  ({len(meta):,} rows)")
    print("Columns:", list(meta.columns))
    # Map filename -> absolute path
    fname_to_path = {os.path.basename(p): p for p in img_files_all}
    meta["abs_path"] = meta["file_name"].map(fname_to_path)
    missing = int(meta["abs_path"].isna().sum())
    print(f"Rows with no matching image: {missing}")
    meta = meta.dropna(subset=["abs_path"]).reset_index(drop=True)
else:
    # Fallback: infer class from parent directory name, patient ID from filename pattern.
    # OCTDL filenames look like: amd_1047099_1.jpg  (class_patient_frame).
    print("Falling back to directory-based class inference.")
    rows = []
    pat = re.compile(r"^([a-zA-Z]+)_(\d+)_", re.IGNORECASE)
    for fp in img_files_all:
        parent = os.path.basename(os.path.dirname(fp)).strip().upper()
        fname = os.path.basename(fp)
        m = pat.match(fname)
        pid = m.group(2) if m else fname.split(".")[0]
        # Disease label: prefer parent-folder name if it's a known class, else the filename prefix
        if parent in {"AMD", "DME", "ERM", "NO", "NORMAL", "RAO", "RVO", "VID", "VMID"}:
            disease = parent
        elif m:
            disease = m.group(1).upper()
        else:
            disease = "UNKNOWN"
        rows.append({
            "file_name": fname,
            "abs_path": fp,
            "disease": disease,
            "patient_id": pid,
            "sex": "",   # not available
            "eye": "",   # not available
        })
    meta = pd.DataFrame(rows)
    print(f"Built fallback metadata: {len(meta):,} rows")
    print("Class distribution from directory inference:")
    print(meta["disease"].value_counts())

# Normalize: 'NO' is OCTDL's name for Normal
meta["disease_norm"] = meta["disease"].str.upper().str.strip().replace({"NO": "NORMAL"})

# Binary referable mapping
def is_referable(d):
    return 0 if str(d).strip().upper() in {"NORMAL", "NO"} else 1
meta["referable"] = meta["disease_norm"].apply(is_referable)

print("\nNative disease (normalized) -> referable mapping:")
print(meta.groupby(["disease_norm", "referable"]).size())
print(f"\nReferable distribution: {meta['referable'].value_counts().to_dict()}")
if "patient_id" in meta.columns and meta["patient_id"].astype(str).str.len().gt(0).any():
    print(f"Unique patients: {meta['patient_id'].nunique():,}")
print(f"\nFinal usable rows: {len(meta):,}")


## Cell 4 — Eval Dataset + transforms

Same eval transform as Kermany (grayscale→3ch, resize 224, ImageNet normalize).

In [ ]:
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class OCTDLDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["abs_path"])
        if self.transform: img = self.transform(img)
        return (img, int(row["referable"]),
                str(row.get("patient_id", "")),
                str(row.get("sex", "")),
                str(row.get("eye", "")))

class KermanyTestDataset(Dataset):
    # In-distribution sanity — reads test split of kermany_index.csv.
    def __init__(self, index_csv, transform=None):
        df = pd.read_csv(index_csv)
        df = df[df["split"] == "test"].reset_index(drop=True)
        df["referable"] = (df["native_class"].str.upper() != "NORMAL").astype(int)
        self.df = df
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"])
        if self.transform: img = self.transform(img)
        return img, int(row["referable"]), str(row["patient_id"]), "", ""

octdl_ds = OCTDLDataset(meta, transform=eval_tf)
kermany_test_ds = KermanyTestDataset(KERMANY_INDEX, transform=eval_tf)
print(f"OCTDL eval: {len(octdl_ds):,} images")
print(f"Kermany in-dist test: {len(kermany_test_ds):,} images")

## Cell 5 — Load the 4-class Kermany model

In [ ]:
KERMANY_CLASSES = ["NORMAL", "CNV", "DME", "DRUSEN"]
NORMAL_IDX = KERMANY_CLASSES.index("NORMAL")

def build_resnet50(num_classes=4, dropout=0.2):
    m = models.resnet50(weights=None)
    in_feat = m.fc.in_features
    m.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_feat, num_classes))
    return m

model = build_resnet50(num_classes=4, dropout=0.2).to(DEVICE)
state = torch.load(CKPT, map_location=DEVICE)
model_state = state["model_state"] if isinstance(state, dict) and "model_state" in state else state
missing, unexpected = model.load_state_dict(model_state, strict=True)
print(f"Loaded {CKPT}")
print(f"  missing keys: {list(missing)}  unexpected keys: {list(unexpected)}")
if isinstance(state, dict):
    print(f"  trained at epoch: {state.get('epoch')}  val_auroc: {state.get('val_auroc')}")
model.eval()

## Cell 6 — Inference

P(referable) = 1 − softmax(NORMAL). Same logic for both datasets.

In [ ]:
@torch.no_grad()
def predict_referable(loader):
    probs, ys, pids, sexes, eyes = [], [], [], [], []
    for batch in loader:
        x, y, pid, sex, eye = batch
        x = x.to(DEVICE, non_blocking=True)
        logits = model(x)
        sm = F.softmax(logits, dim=1)
        p_ref = (1.0 - sm[:, NORMAL_IDX]).cpu().numpy()
        probs.append(p_ref)
        ys.append(np.asarray(y))
        pids.extend(list(pid)); sexes.extend(list(sex)); eyes.extend(list(eye))
    return (np.concatenate(probs), np.concatenate(ys),
            np.array(pids), np.array(sexes), np.array(eyes))

BS = 128
octdl_loader = DataLoader(octdl_ds, batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)
kermany_loader = DataLoader(kermany_test_ds, batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)

print("OCTDL transfer inference...")
t0 = time.time()
od_p, od_y, od_pid, od_sex, od_eye = predict_referable(octdl_loader)
print(f"  {len(od_y):,} predictions in {time.time()-t0:.1f}s.  pos_rate={od_y.mean():.3f}")

print("\nKermany in-distribution inference...")
t0 = time.time()
km_p, km_y, _, _, _ = predict_referable(kermany_loader)
print(f"  {len(km_y):,} predictions in {time.time()-t0:.1f}s.  pos_rate={km_y.mean():.3f}")

## Cell 7 — Metrics with bootstrap 95% CIs

In [ ]:
def point_metrics(p, y):
    auroc = roc_auc_score(y, p)
    auprc = average_precision_score(y, p)
    fpr, tpr, _ = roc_curve(y, p)
    idx95 = int(np.argmin(np.abs((1 - fpr) - 0.95)))
    j_idx = int(np.argmax(tpr - fpr))
    return dict(
        n=int(len(y)), positive_rate=float(y.mean()),
        auroc=float(auroc), auprc=float(auprc),
        sensitivity_at_95_specificity=float(tpr[idx95]),
        balanced_sensitivity=float(tpr[j_idx]),
        balanced_specificity=float(1 - fpr[j_idx]),
    )

def bootstrap_ci(p, y, n_boot=1000, alpha=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y)
    aurocs, auprcs, sens95s = [], [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yi, pi = y[idx], p[idx]
        if yi.min() == yi.max():
            continue
        aurocs.append(roc_auc_score(yi, pi))
        auprcs.append(average_precision_score(yi, pi))
        fpr, tpr, _ = roc_curve(yi, pi)
        sens95s.append(float(tpr[int(np.argmin(np.abs((1 - fpr) - 0.95)))]))
    def ci(v):
        v = np.asarray(v)
        return [float(np.quantile(v, alpha/2)), float(np.quantile(v, 1-alpha/2))]
    return dict(auroc_ci95=ci(aurocs), auprc_ci95=ci(auprcs),
                sensitivity_at_95_specificity_ci95=ci(sens95s))

def full_eval(name, p, y):
    pt = point_metrics(p, y); ci = bootstrap_ci(p, y)
    res = {"dataset": name, **pt, **ci}
    print(f"\n=== {name} ===")
    print(json.dumps(res, indent=2))
    return res

results = {}
results["octdl_transfer"] = full_eval("OCTDL transfer (binary referable)", od_p, od_y)
results["kermany_in_dist"] = full_eval("Kermany test in-dist (binary referable)", km_p, km_y)

## Cell 8 — Mini fairness audit on OCTDL

Subgroup AUROC by `sex` and `eye`. Sample sizes are small per subgroup so CIs will be wide. This is a proof-of-concept for the real BRSET audit.

In [ ]:
def subgroup_eval(p, y, group_arr, group_name):
    print(f"\n--- Subgroup analysis by {group_name} ---")
    rows = []
    for g in sorted(set(group_arr.tolist())):
        mask = group_arr == g
        if mask.sum() < 30:
            print(f"  {group_name}={g!r}: n={int(mask.sum())} — skipped (too few)")
            continue
        pg, yg = p[mask], y[mask]
        if yg.min() == yg.max():
            print(f"  {group_name}={g!r}: n={int(mask.sum())} — skipped (single-class)")
            continue
        m = point_metrics(pg, yg)
        ci = bootstrap_ci(pg, yg, n_boot=500)
        rows.append({"group": g, **m, **ci})
        print(f"  {group_name}={g!r:10s}  n={m['n']:4d}  pos_rate={m['positive_rate']:.3f}  "
              f"AUROC={m['auroc']:.4f}  CI={ci['auroc_ci95']}  "
              f"Sens@95Spec={m['sensitivity_at_95_specificity']:.3f}")
    return rows

results["octdl_by_sex"] = subgroup_eval(od_p, od_y, od_sex, "sex")
results["octdl_by_eye"] = subgroup_eval(od_p, od_y, od_eye, "eye")

## Cell 9 — Summary table + persist

In [ ]:
OUT = pathlib.Path("/kaggle/working/results")
OUT.mkdir(exist_ok=True)

summary = {
    "experiment": "kermany_resnet50_baseline_transferred_to_octdl",
    "checkpoint": CKPT,
    "label_scheme": "binary_referable",
    "headline_auroc_drop": (results["kermany_in_dist"]["auroc"]
                            - results["octdl_transfer"]["auroc"]),
    "headline_sens95_drop": (results["kermany_in_dist"]["sensitivity_at_95_specificity"]
                             - results["octdl_transfer"]["sensitivity_at_95_specificity"]),
    "results": results,
}

with open(OUT / "crossdataset_metrics.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"Wrote {OUT/'crossdataset_metrics.json'}")

def fmt(x): return f"{x:.4f}"
def fmtci(c): return f"[{c[0]:.3f}, {c[1]:.3f}]"

km = results["kermany_in_dist"]; od = results["octdl_transfer"]
table = (
    "\n| Dataset                | n      | AUROC  | AUROC 95% CI      | Sens@95Spec |\n"
    "|------------------------|--------|--------|--------------------|-------------|\n"
    f"| Kermany (in-dist test) | {km['n']:6d} | {fmt(km['auroc'])} | {fmtci(km['auroc_ci95'])} | {fmt(km['sensitivity_at_95_specificity'])} |\n"
    f"| OCTDL  (transfer)      | {od['n']:6d} | {fmt(od['auroc'])} | {fmtci(od['auroc_ci95'])} | {fmt(od['sensitivity_at_95_specificity'])} |\n"
    f"\n**Δ AUROC (in-dist − transfer): {summary['headline_auroc_drop']:+.4f}**\n"
)
print(table)
(OUT / "summary.md").write_text(table)

## Cell 10 — Persist artifacts off Kaggle

In [ ]:
import shutil
zip_path = pathlib.Path("/kaggle/working/kermany_to_octdl_results.zip")
if zip_path.exists(): zip_path.unlink()
shutil.make_archive(zip_path.with_suffix(""), "zip", root_dir=str(OUT))
print(f"Wrote {zip_path}  ({zip_path.stat().st_size/1e6:.2f} MB)")
print("\nDownload from Notebook page → Output tab → kermany_to_octdl_results.zip")